<a href="https://colab.research.google.com/github/VladaShudegova/932121.shudegova.vlada.lab1/blob/main/%D0%9E%D1%82%D1%87%D0%B5%D1%82_%D0%BF%D0%BE_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B5_Angr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Для анализа я использую фреймворк **angr**, который работает на уровне машинного кода и позволяет выполнять бинарник символически.

В данном фреймворке используются следующие сущности:

**Project** — загружает бинарный файл через свой загрузчик CLE в виртуальное адресное пространство: секции, сегменты, точки входа, таблицу функций.

**SimulationManager (simgr)** — симулятор кода, который выделяет набор состояний и определяет механикау шагания/деления по stash’ам.​
    
**SimState** — снимок программы на конкретном шаге, который знает о себе регистры, память, стек, constraints.
    
**CFG (CFGFast/CFGAccurate)** — граф. Вершинами являются базовые блоки или функции. Рёбрами являются возможные переходы управления (следующая инструкция, переход по `if`, переход по `goto`/`jmp`, возврат из функции).

**Solver (state.solver)** — интерфейс к SMT: добавление/упрощение/проверка constraints, извлечение моделей.
    

In [3]:
!pip install angr

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.5/49.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.2/141.2 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.9/13.9 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.3/59.3 kB 3.9 MB/s eta 0:0

In [4]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [5]:
import angr
import claripy

## Поэтапная обработка бинарного файла(ELF)

### Для начала загружаю бинарный файл в виртуальное пространство.

In [15]:
import angr, claripy

proj = angr.Project("/content/gdrive/MyDrive/vkr/elf/bank2.elf", auto_load_libs=False)

# адрес точки входа entry моего бинарника
main_obj = proj.loader.main_object

print("entry:", hex(main_obj.entry))

for s in main_obj.sections:
    print(hex(s.vaddr), s.name, s.is_executable, main_obj.find_section_containing(s.vaddr))


entry: 0x400078
0x0  False None
0x400078 .text True <.text | offset 0x78, vaddr 0x400078, size 0x49>
0x4000c8 .eh_frame False <.eh_frame | offset 0xc8, vaddr 0x4000c8, size 0x38>
0x0 .comment False None
0x0 .symtab False None
0x0 .strtab False None
0x0 .shstrtab False None


Получаю список секций (.text, .data, .bss) с адресами и флагами исполнения. Через `find_section_containing(addr)` могу по любому адресу понять, в каком участке моего ELF он находится.

### Построение  **Control Flow Graph (CFG)**

CFG даёт мне «скелет» программы: какие есть функции, как они связаны, по каким адресам расположены ветвления и циклы.

In [25]:
cfg = proj.analyses.CFGFast()  # строю CFG именно для конкретного ELF

print("functions total:", len(cfg.kb.functions))

main_func = cfg.kb.functions.function(name="main")  # нахожу main
print("main at:", hex(main_func.addr))

for block in main_func.blocks:
    print()
    print(block, "  block:", hex(block.addr),  "size:", block.size, "translate type:", block.vex.jumpkind)

    for insn in block.capstone.insns:
      # дизассемблированный код этого блока
      print(hex(insn.address), insn.mnemonic, insn.op_str)



functions total: 1
main at: 0x400078

<Block for 0x400078, 16 bytes>   block: 0x400078 size: 16 translate type: Ijk_Boring
0x400078 push rbp
0x400079 mov rbp, rsp
0x40007c mov dword ptr [rbp - 4], edi
0x40007f mov dword ptr [rbp - 8], esi
0x400082 cmp dword ptr [rbp - 4], 0
0x400086 jle 0x40008e

<Block for 0x40008e, 7 bytes>   block: 0x40008e size: 7 translate type: Ijk_Boring
0x40008e mov eax, 1
0x400093 jmp 0x4000bf

<Block for 0x400088, 6 bytes>   block: 0x400088 size: 6 translate type: Ijk_Boring
0x400088 cmp dword ptr [rbp - 8], 0x11
0x40008c jg 0x400095

<Block for 0x4000bf, 2 bytes>   block: 0x4000bf size: 2 translate type: Ijk_Ret
0x4000bf pop rbp
0x4000c0 ret 

<Block for 0x400095, 14 bytes>   block: 0x400095 size: 14 translate type: Ijk_Boring
0x400095 mov eax, dword ptr [rbp - 4]
0x400098 cdq 
0x400099 idiv dword ptr [rbp - 8]
0x40009c cmp eax, 0x3e7
0x4000a1 jg 0x4000aa

<Block for 0x4000a3, 7 bytes>   block: 0x4000a3 size: 7 translate type: Ijk_Boring
0x4000a3 mov eax, 2


Angr разложил код main на базовые блоки с конкретными адресами. Данные адреса я уже использую, сопоставляя с `state.addr` при символическом выполнении.

Также из данных блоков можно получить дизассемблированный код или VEX-представление(IR), тип перехода(обычный `Ijk_Boring`, вызов `Ijk_Call`, возврат `Ijk_Ret`, syscall ). Также можно найти соседей в графе функции: список последующих блоков, список предшественников.

На основе `jumpkind` и числа successors можно понять, что это за точка: линейный блок без ветвлений, условное ветвление (2 successors),возврат (`Ijk_Ret`, 0 successors), вызов функции (`Ijk_Call` + edge к вызванной функции).


### Символьный запуск main и получение constraints

В elf main принимает balance и age — я cделаю их символическими:

In [32]:
import claripy

balance = claripy.BVS("balance", 32)
age     = claripy.BVS("age", 32)

state = proj.factory.call_state(main_func.addr, balance, age)  # вызов main

# добавляю глобальные ограничения на входы
state.solver.add(balance >= 0)
state.solver.add(age >= 0)
state.solver.add(age <= 120)

simgr = proj.factory.simgr(state)
simgr.run()  # прохожу все пути по коду main


print("deadended states:", len(simgr.deadended))


deadended states: 5


Теперь в **simgr** у меня несколько состояний, каждое соответствует конкретному пути по коду. Для любого такого состояния я могу получить ограничения:

In [33]:
for st in simgr.active + simgr.deadended:
    print()
    print("addr:", hex(st.addr))
    print("constraints:")
    for c in st.solver.constraints:
        print("  ", c)



addr: 0x500038
constraints:
   <Bool balance_20_32 >= 0x0>
   <Bool age_21_32 >= 0x0>
   <Bool age_21_32 <= 0x78>
   <Bool balance_20_32 <=s 0x0>

addr: 0x500038
constraints:
   <Bool balance_20_32 >= 0x0>
   <Bool age_21_32 >= 0x0>
   <Bool age_21_32 <= 0x78>
   <Bool balance_20_32 >s 0x0>
   <Bool age_21_32 <=s 0x11>

addr: 0x500038
constraints:
   <Bool balance_20_32 >= 0x0>
   <Bool age_21_32 >= 0x0>
   <Bool age_21_32 <= 0x78>
   <Bool balance_20_32 >s 0x0>
   <Bool age_21_32 >s 0x11>
   <Bool age_21_32 != 0x0>
   <Bool (balance_20_32 >> 0x1f .. balance_20_32) /s SignExt(32, age_21_32)[31:0] <=s 0x3e7>

addr: 0x500038
constraints:
   <Bool balance_20_32 >= 0x0>
   <Bool age_21_32 >= 0x0>
   <Bool age_21_32 <= 0x78>
   <Bool balance_20_32 >s 0x0>
   <Bool age_21_32 >s 0x11>
   <Bool age_21_32 != 0x0>
   <Bool (balance_20_32 >> 0x1f .. balance_20_32) /s SignExt(32, age_21_32)[31:0] >s 0x3e7>
   <Bool balance_20_32 <=s 0xf4240>

addr: 0x500038
constraints:
   <Bool balance_20_32 >= 

 Логические условия представлены в виде SMT‑ограничений. Я могу для каждого пути проверить выполнимость(`st.solver.satisfiable()`), получить пример входов( `st.solver.eval(balance)`, `st.solver.eval(age)`.

 Могу добавить доплнительные ограничения на символьные переменные(`st.solver.add(cond)`).

​

In [34]:
for i, st in enumerate(simgr.deadended):
    print(f"\n=== state {i} ===")
    # проверяю выполнимость текущих ограничений
    sat = st.solver.satisfiable()
    print("satisfiable:", sat)
    if not sat:
        continue

    # получаю пример входов для этого пути
    ex_balance = st.solver.eval(balance)
    ex_age     = st.solver.eval(age)

    # получаю значение, которое вернула main (eax)
    ret_val = st.solver.eval(st.regs.eax)

    print(f"example inputs: balance = {ex_balance}, age = {ex_age}")
    print(f"return value (eax): {ret_val}")



=== state 0 ===
satisfiable: True
example inputs: balance = 0, age = 0
return value (eax): 1

=== state 1 ===
satisfiable: True
example inputs: balance = 1, age = 0
return value (eax): 1

=== state 2 ===
satisfiable: True
example inputs: balance = 8056, age = 65
return value (eax): 2

=== state 3 ===
satisfiable: True
example inputs: balance = 485365, age = 119
return value (eax): 0

=== state 4 ===
satisfiable: True
example inputs: balance = 77714754, age = 71
return value (eax): 3


### Существующие инструменты в Angr для решения проблемы взрыва путей

Сам SimulationManager хранит множество состояний (SimState) и управляет их развитием по стэшам: active, deadended, errored, unconstrained, found и т.д.

 Exploration Techniques позволяет настраивать поведение симуляции. Он может ограничить глубину, отсечь бесконечные или слишком длинные циклы, фокусироваться на путях, ведущих к заданным адресам (find/avoid).

 Базовый класс **ExplorationTechnique** предоставляет точки расширения: step, filter, selector и др. Документация прямо отмечает, что angr «предоставляет отличную базу для реализации новых техник контроля path explosion», и большинство подходов удобно реализуются именно как ExplorationTechnique.

 Существующие техники в angr используют в основном структурные эвристики: глубина пути, наличие циклов, целевые адреса. Они почти не смотрят внутрь самих ограничений, которые накапливаются в каждом SimState.

Фреймворк позволяет анализировать ограничения на кажом состоянии симуляции и классифицировать пути. На это я и хочу сделать упор.

### Трудности во время работы с разными типами бинарных файлов

Для учебного проекта я целенаправленно выбираю **ELF‑бинарники под x86‑64 Linux**.

Дело в том, что на ELF корректно работают CFG‑анализы, вызовы функций и большинство встроенных симуляций библиотек (SimProcedures). Сами сегменты, точки входа корректно отображаются.

Для Mach‑O сами разработчики отмечают, что поддержка ограничена: особенности формата, секции и символы обрабатываются менее надёжно.

Для PE под Windows приходится дополнительно учитывать загрузку DLL, специфические таблицы импорта/экспорта и поведение ОС; это вносит много интеграционных проблем, не связанных с моей исследовательской задачей по constraints и path explosion.

